<a href="https://colab.research.google.com/github/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FlyRank ML Internship: Your Dataset and Lane Guide

## Welcome to the Applied Search Intelligence Track

This notebook is your interactive guide to the FlyRank ML internship. It walks you through:

1. **Understanding the Core Idea** - What you'll be doing
2. **Exploring Your Data** - The warehouse release and starter dataset
3. **Choosing Your Lane** - Which project direction to take
4. **Getting Started** - First steps with the data
5. **Avoiding Common Traps** - What not to do

Read this together with:
- `docs/ml-core-foundation-framework.md` (deep reference)
- `docs/intern-free-tooling-guide.md` (tooling guide)
- The week-by-week curriculum on your portal board
- The dataset manifest on Hugging Face

**Remember:** Your goal is not to memorize FlyRank's existing rules. Your goal is to understand the problem, look at the evidence, build a simple starting point, test a better method, check it honestly, and turn the result into a ranked list of safe recommendations.

## 0. Where These Numbers Come From

Every count and date in this guide was checked against the real data pipeline. The release was built, scrambled, and verified on FlyRank's side before publication.

### The Release
- **Release**: [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse) on Hugging Face
- **Build ID**: `flyrank_pseudonymized_warehouse_release_v20260703`
- **Export date**: `2026-07-03`
- **Freshness lag**: Data stops at `2026-06-30` (3 days cut off intentionally)
- **Format**: Parquet files, 10,000 rows per chunk

### Important Rules
- **For warehouse work, use the Hugging Face release** - it's a fixed snapshot
- **Stick to this release** - results built on it can be compared and rerun
- **Never use raw BigQuery credentials, raw client names, raw domains, raw URLs, raw private queries, or raw text** - everything sensitive is scrambled

## 1. The Core Idea

The whole track follows one workflow:

```
research question ? safe data contract ? signal audit ? baseline score ? your chosen lane ? validation ? ranked action recommendations
```

### What These Words Mean

| Term | Meaning |
|------|---------|
| **Research question** | The one question your project answers |
| **Data contract** | A short written promise about which data you will use and how |
| **Signal audit** | A check of whether the columns actually contain useful information |
| **Baseline** | The simplest possible method, built first |
| **Validation** | The test that proves your result is real |
| **Ranked action recommendations** | The final output: a list of pages ordered by priority |

### Teaching Method

```
core idea first ? AI-assisted implementation second ? human judgment last
```

That means:
- Start with the content and search problem, not with a model
- Use AI to draft code, ideas, checks, and explanations - but never let AI decide what is true
- Treat product scores and flags as useful background, not as the truth
- Prefer simple methods you can explain, until something harder is clearly worth it
- Tie every claim to something the data can actually prove

## 2. Which Data to Use

### The Warehouse Tables

The warehouse release gives you **dimension tables** (describes things) and **fact tables** (records events over time).

| Table | Rows | Grain | Main Use |
|-------|------|-------|----------|
| `dim_clients` | 104 | one row per pseudonymized client | grouping, client-level checks |
| `dim_content` | 519,606 | one row per pseudonymized content item | content metadata and joins |
| `fact_content_daily_performance` | 78,835,655 | daily � client � content | time-series features, trend labels |
| `fact_content_query_90d` | 2,414,248 | client � content � query hash | query-mix features |

### Which Lanes Are Ready, and With What Data

A **lane** is a project direction you can pick:

| Lane | Type | Default Dataset | Output |
|------|------|-----------------|--------|
| Ranking Signal Analysis | Core lane | warehouse or starter | Signal report and recommendations |
| Refresh/Content Opportunity Scoring | Core lane | starter + warehouse support | Ranked review queue with scores |
| Structured Content Archetype Clustering | Core lane | warehouse or starter | Cluster profiles and action mapping |
| CTR/Engagement Opportunity Scoring | Core lane | warehouse or starter | Ranked opportunity score |
| AI Referral Opportunity | Freestyle | warehouse daily facts | EDA/ranking only |
| Growth/Recovery/Momentum Prediction | Freestyle | warehouse daily facts | Future-window model |

## 3. Field Types

Sort every column into one of these buckets before you model anything:

| Field Type | Meaning | Example Fields | Default Use |
|------------|---------|----------------|-------------|
| **Join keys** | Scrambled ids that connect tables | `client_hash_id`, `content_hash_id` | Join and group only |
| **Observed signals** | Real measurements from search/analytics | impressions, clicks, position, CTR, sessions | Good candidate features |
| **Derived measurements** | Numbers calculated from observed signals | `trend_pct`, age/freshness tiers | Usually fine |
| **Product context** | Rule-outputs FlyRank's app computes | `health_score`, `priority_score` | NOT in your dataset |
| **Target/proxy fields** | The thing your model predicts | future decline, recovery, top-K priority | Define these yourself |
| **Raw-origin context** | Real search queries, URLs, titles | raw query/URL/title/client fields | SCRAMBLED, never in your data |

## 4. Observable Signals, Not Product Decisions

FlyRank's product computes rule-based decision flags and combined scores - things like `health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, and `action_type`.

**These product decisions are deliberately NOT in your data.**

Your data ships observable search and engagement signals, plus transparent derived buckets:
- tiers, `trend_direction`, `trend_pct`, CTR, rates
- and nothing else

This is on purpose, so you discover signal from evidence instead of accidentally copying the product's own answers.

### Why This Matters

If you feed the product's own decision into your model, the model just learns to copy that decision. The score looks great, but you discovered nothing - the answer was already in the input. This trap is called a **circular result**.

### What to Do Instead

Build only from observable signals (things that were true BEFORE anyone decided anything), and let your model find its own signal.

The strongest labels for supervised learning come from **future observed outcomes**, not from a current decision:
- Traffic that later declined
- A page that later recovered
- CTR that later changed
- Engagement that later changed

## 5. The Starter Playground: What It Proves

The starter playground is a runnable example, not the whole internship.

### Main Input
```text
data/raw/content_refresh_anonymized.csv
```

### The Pipeline
```text
01_prepare_features.py ? 02_baseline_score.py ? 03_train_model.py ? 04_evaluate_and_export.py ? 05_build_pdf_report.py
```

### Starter Target
```text
is_declining_label = trend_direction == "down"
```

This is simple on purpose. It shows the workflow end to end, but notice its weakness: it's a bucket calculated from the current window, not a future outcome. Treat it as a beginner **proxy label**.

### Starter Model Results (Verified)

| Method | ROC AUC | Average Precision | Precision@50 |
|--------|---------|-------------------|--------------|
| baseline rules | 0.627 | 0.468 | 0.240 |
| logistic regression | 0.700 | 0.522 | 0.400 |
| decision tree | 0.742 | 0.575 | 0.540 |
| random forest | 0.750 | 0.618 | 0.740 |

**What this means in plain words:**
- **Precision@50**: of the top 50 pages the system says to review first, how many actually turned out positive?
- The baseline got about 12 of its top 50 right: `0.240 * 50`
- The random forest got about 37 of its top 50 right: `0.740 * 50`
- Strong evidence that a learned ranking can beat a fixed rule

## 6. What "The Right Page to Fix" Means

In this internship, "the right page to fix" usually means:

```text
the right page to REVIEW FIRST, based on evidence and limited capacity
```

It does NOT mean:

```text
a page guaranteed to recover if someone edits it
```

To prove that a refresh CAUSED a recovery, you'd need an experiment or causal design - something this data alone cannot give.

### A Good Candidate Page Usually Has:

- ? Enough demand or exposure to matter
- ? Evidence of movement, weakness, or opportunity
- ? An action someone could realistically take
- ? Reason codes a reviewer can inspect
- ? No obvious leakage or private-data issue
- ? No obvious consolidation, seasonality, or noise explanation

## 7. Decline Versus Consolidation, Seasonality, and Noise

Not every drop in traffic is a decline. Before you label a page "declining," rule out the look-alikes:

| Pattern | What It Means | How to Check |
|---------|---------------|--------------|
| **Real decline** | The same content keeps losing visibility | Compare earlier window to later window |
| **Consolidation** | One URL drops because a related URL absorbed demand | Check if a sibling page gained |
| **Seasonality** | Demand naturally drops with calendar/market | Compare to site-level trends |
| **SERP or AI click loss** | Impressions hold while clicks drop | Look at each metric separately |
| **Noise** | Small, low-volume wiggles with no pattern | Require minimum volume and persistence |

A strong definition spells out:
- **Magnitude**: drop bigger than a threshold
- **Window**: prior 90 days versus next 30 days
- **Persistence**: drop lasts longer than a short blip
- **Minimum volume**: enough impressions or sessions
- **Group checks**: related pages didn't absorb the traffic
- **Prediction-time discipline**: feature window never overlaps target window

## 8. Lane Guide

Whichever lane you pick, you produce the same core artifacts:
1. Data contract
2. Signal audit
3. Baseline
4. Model or analysis
5. Validation
6. Ranked action output
7. Public-safe write-up

### Lane 1: Ranking Signal Analysis

**Question:**
```text
Which safe content and search signals are associated with visibility, clicks, engagement, or movement?
```

**Good Data:**
- Warehouse release (`dim_content` + `fact_content_daily_performance`)
- Starter dataset

**Good Methods:**
- EDA (exploratory data analysis)
- Correlations and grouped summaries
- Simple regressions or classification (if you define a target)
- Feature importance from a simple model
- Effect sizes (not just "is there a link" but "how big is it")

**Output:**
- Signal report with charts
- Practical content recommendations
- Clear caveats (observational results)

**Common Mistakes:**
- ? Claiming you proved a Google algorithm factor
- ? Using a score as both feature and target
- ? Reading big meaning into weak correlations

### Lane 2: Refresh / Content Opportunity Scoring

**Question:**
```text
Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?
```

**Good Data:**
- Warehouse release or starter dataset
- Daily table for stronger time-window labels

**Useful Baseline Ideas:**
- Stale visible page
- Declining with demand
- Thin visible page
- Page-one decay risk
- CTR or engagement review context

**Output:**
- Ranked action queue
- Suggested action per page
- Reason codes
- Confidence label
- Model/analysis card

**Common Mistakes:**
- ? Treating "declining" as a guarantee a refresh will pay off
- ? Using metrics from the future window as features
- ? Hiding the reason behind a high score

### Lane 3: Structured Content Archetype Clustering

**Question:**
```text
What performance archetypes exist across the content inventory?
```

An **archetype** is a recurring "type" of page - clustering finds groups that behave alike.

**Good Data:**
- Warehouse release or starter dataset

**Good Methods:**
- Scaling numeric features
- K-Means or another clustering method you can explain
- PCA or another 2D projection for visualization
- Cluster profiling (describe each cluster with its typical numbers)

**Possible Archetypes:**
- Champions
- Rising stars
- Hidden gems
- Stale visible pages
- Weak/no-demand pages
- Engagement-problem pages
- Cannibalization-risk pages

**Output:**
- Cluster profiles
- Archetype names
- Action mapping (protect, improve, rewrite, merge, prune, monitor)

**Common Mistakes:**
- ? Naming clusters before inspecting them
- ? Treating clusters as true labels
- ? Using unsafe text fields or raw URLs
- ? Calling metric/token-count clustering "semantic clustering"

### Lane 4: CTR / Engagement Opportunity Scoring

**Question:**
```text
Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?
```

**Good Data:**
- Warehouse release or starter dataset
- Daily table

**Good Features:**
- Impressions, clicks, CTR, average position
- Position tier, intent/content type
- Age and freshness
- Sessions and engagement context

**Good Methods:**
- Expected CTR by position tier (compare pages only to others in same tier)
- Residual or gap analysis
- Ranked scoring
- Classification (if you define a leakage-safe label)

**Output:**
- Ranked list of CTR/engagement review candidates
- Reason codes (high impressions, low CTR, strong position, etc.)
- Action suggestions (rewrite title/meta, improve intent match, etc.)

**Common Mistakes:**
- ? Comparing CTR across different positions without adjusting
- ? Ignoring low-volume noise
- ? Assuming low CTR always means title/meta is bad

## 9. Freestyle - Your Own Question

Freestyle means you skip the four lanes and bring your own search or discoverability question: your own joins, your own features, your own labels, and your own model, built on the full warehouse.

**No approval, no gate** - freestyle is a normal choice.

### Freestyle Direction: AI Referral Opportunity

**Question:**
```text
What broad content patterns appear around AI-referred traffic, and where might AI visibility be improved?
```

**Data:** `sessions_ai` in `fact_content_daily_performance`

**Best Use:**
- EDA
- Broad pattern analysis
- Opportunity ranking
- Careful written discussion of what this data can/cannot say

**Big Warning:**
AI-referral sessions are **sparse** - remember the density table: 30,177 rows with AI sessions against 78.8 million daily rows.

**Common Mistakes:**
- ? Treating "no AI sessions" as proof AI platforms can't understand content
- ? Making strong claims from sparse data
- ? Building binary classifiers without valid positive/negative examples
- ? Claiming AI citations, rankings, or visibility

### Freestyle Direction: Growth / Recovery / Momentum Prediction

**Question:**
```text
Can we predict which pages are likely to decline, recover, or gain momentum?
```

**Data:** Daily windows you build yourself from `fact_content_daily_performance`

**The Label Shape That Keeps You Honest:**
```
prior feature window ? future target window
```

Examples:
- Prior 90 days of features ? next 30 days decline
- Prior 28 days of features ? next 28 days growth

**Big Warning:** This direction lives or dies on **clean future-window labels and strict leakage control**.

**Good Methods:**
- Time-aware or client-grouped validation
- Logistic regression, tree, random forest, or gradient boosting
- Calibration and threshold review
- Precision@K for review queues

**Common Mistakes:**
- ? Using target-window metrics as features
- ? Letting pages from same client land in both train and test
- ? Calling seasonal movement "model skill"

## 10. Do Not Do This

- ? Train an AI-session classifier on the sparse AI-session data alone
- ? Rebuild a product decision flag and use it as an ordinary model feature
- ? Publish or try to reconstruct raw query, URL, title, domain, client, category, or keyword examples
- ? Claim a refresh caused a recovery unless you ran an explicit experiment
- ? Claim Google algorithm factors, AI citations, or AI rankings from these datasets

## 11. Thresholds and Decision Policies

A **threshold** is a cutoff you choose - and every threshold is a policy choice, not a universal truth.

### Examples of Thresholds You'll Have to Pick

- How many impressions are enough to matter?
- How big a drop counts as decline?
- Where CTR becomes weak for a position tier?
- How many pages fit the review team's capacity?
- What score earns the label "high confidence"?

### A Good Way to Pick One

1. Start with a simple rule you can explain
2. Show how many rows it captures
3. Review real examples from top, middle, and edge
4. Try alternative thresholds and watch what changes
5. Compare precision@K, recall, false positives
6. Pick the threshold that matches the real decision capacity and risk
7. Write down the trade-off you accepted

### For Ranked Action Work

Top-K metrics usually beat generic accuracy:

| Metric | Use When |
|--------|----------|
| Precision@20 | Reviewer checks 20 pages |
| Precision@50 | Team can act on 50 candidates |
| Average precision | The whole ranking matters |
| Recall | Missing a true problem is expensive |

## 12. Validation Rules

A model or analysis is only worth something if its validation design matches the problem.

### Pick the Design on Purpose

| Design | When to Use |
|--------|-------------|
| Plain train/test split | Examples are truly independent |
| Client/group holdout | Pages from same client share patterns |
| Time-aware split | Predicting future movement |
| Top-K review | Ranked queues |
| Cluster stability checks | Archetype lanes |
| Leakage audit | Every lane, every time |

### The Leakage Checklist - Ask Each Question Out Loud

- [ ] Are any features calculated after the decision point?
- [ ] Does the feature window overlap the target window?
- [ ] If you rebuilt any product output, did it slip in as a normal feature?
- [ ] Does a derived field secretly encode the target?
- [ ] Are duplicate or related rows split across train/test in a way that makes the test too easy?
- [ ] Are you testing on clients or time periods the model has not effectively already seen?

## 13. Precalculated Columns: How to Use Them

Precalculated columns (like tiers and trend buckets) are allowed and useful. They should make the problem easier to understand - not do the thinking for you.

### Good Uses

- Understand how FlyRank's existing system frames a page
- Build a transparent baseline
- Compare your model against the existing rule
- Inspect where your model disagrees with the product's framing
- Explain reason codes in your final action queue

### Risky Uses

- Rebuilding a product decision flag and then feeding it to a discovery model as a feature
- Calling a product flag "the truth" without checking future outcomes
- Optimizing your model just to agree with the old rule
- Leaning on a combined score instead of explaining the underlying signals

### Five Questions to Ask First

1. What is it trying to measure?
2. Was it available at prediction time?
3. Is it context, a feature, a label, or an output?
4. Could it leak the answer?
5. Does your result still hold if you remove it?

## 14. Public-Safe Output Rules

Your final deliverable is a deployed research paper - a public web page anyone can read - plus your repo. Because it's public, everything in it must be public-safe.

### Allowed

? Pseudonymized IDs
? Aggregated metrics
? Charts built from safe data
? High-level examples
? Careful observed/directional claims ("we observed", "this suggests")
? Generic content actions

### Not Allowed

? Client names
? Domains
? URLs from the data
? Raw private queries
? Titles or text fields that could reveal who a client is
? Credentials or BigQuery internals
? Claims that you proved Google's algorithm
? Claims that a refresh caused a recovery (unless you ran a valid causal design)

## 15. What Done Looks Like: The Capstone Self-Check

Your capstone ships as two things:

1. **Deployed research paper** (public web page)
2. **Repo** with `submission/paper_url.txt` at the root holding one line - the direct URL of your deployed paper

### Before You Call It Done

Check that your work answers every one of these questions:

- [ ] What problem are you solving?
- [ ] What decision or ranking are you improving?
- [ ] What does one row mean - what is your grain?
- [ ] What data tables did you use?
- [ ] Which fields are features, labels/proxies, context, excluded, or leakage risks?
- [ ] What baseline did you build?
- [ ] What model or analysis did you choose, and why?
- [ ] What split or validation design did you use?
- [ ] What metric matches the real decision?
- [ ] Did the model beat the baseline? If not, what did you learn?
- [ ] What are the top recommendations?
- [ ] What reason codes explain them?
- [ ] What would make a recommendation wrong?
- [ ] Which claims are safe, and which are not proven?
- [ ] Could someone else rerun your work from your repo alone?

## 16. One-Sentence Mental Model

You are not building magic SEO automation. You are using safe real-world search and content data to learn which signals help prioritize content decisions, then checking honestly whether that prioritization beats a transparent rule - and saying clearly where human review is still required.

## Getting Started: Quick Setup

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

### Load the Starter Dataset

In [ ]:
# Load the starter dataset
starter_path = Path('data/raw/content_refresh_anonymized.csv')
if starter_path.exists():
    df_starter = pd.read_csv(starter_path)
    print(f"? Loaded starter dataset: {len(df_starter):,} rows")
    print(f"Columns: {df_starter.columns.tolist()}")
    print("\nFirst 5 rows:")
    display(df_starter.head())
else:
    print("?? Starter dataset not found. Run the pipeline first with `python scripts/run_all.py`")

### Explore the Data

In [ ]:
# Basic data exploration
print("=== Data Overview ===")
print(f"Rows: {len(df_starter):,}")
print(f"Columns: {len(df_starter.columns)}")
print("\n=== Data Types ===")
print(df_starter.dtypes.value_counts())

print("\n=== Missing Values ===")
missing = df_starter.isnull().sum()
missing = missing[missing > 0]
print(missing)

### Check Trend Distribution

In [ ]:
# Check the target variable distribution
if 'trend_direction' in df_starter.columns:
    trend_counts = df_starter['trend_direction'].value_counts()
    print("=== Trend Direction Distribution ===")
    print(trend_counts)
    print(f"\nPercentage 'down': {trend_counts.get('down', 0) / len(df_starter) * 100:.1f}%")

    # Visualize
    plt.figure(figsize=(8, 5))
    trend_counts.plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA94D'])
    plt.title('Trend Direction Distribution in Starter Dataset')
    plt.xlabel('Trend Direction')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### Check Key Metrics

In [ ]:
# Summary statistics for key metrics
key_metrics = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'content_age_days']
available_metrics = [m for m in key_metrics if m in df_starter.columns]

print("=== Key Metrics Summary ===")
display(df_starter[available_metrics].describe())

### Which Lane Should You Choose?

Based on your interests and the data available:

In [ ]:
def suggest_lane(df):
    """Suggest a lane based on data characteristics"""
    suggestions = []

    # Check if we have enough data for each lane
    if len(df) > 10000:
        suggestions.append(("Ranking Signal Analysis", "Large dataset perfect for correlation analysis"))

    if 'trend_direction' in df.columns and df['trend_direction'].notna().sum() > 1000:
        suggestions.append(("Refresh/Content Opportunity Scoring", "Trend data available for labeling"))

    if 'content_type' in df.columns and 'word_count' in df.columns:
        suggestions.append(("Structured Content Archetype Clustering", "Content features available"))

    if 'ctr' in df.columns and 'avg_position' in df.columns:
        suggestions.append(("CTR/Engagement Opportunity Scoring", "CTR and position data available"))

    return suggestions

print("=== Suggested Lanes Based on Your Data ===")
for i, (lane, reason) in enumerate(suggest_lane(df_starter), 1):
    print(f"{i}. {lane}: {reason}")

## Next Steps

1. **Choose your lane** from the suggestions above
2. **Read the week-by-week curriculum** on your portal board
3. **Set up the Hugging Face dataset** (request access)
4. **Start your first notebook** in `work/notebooks/`
5. **Run the starter pipeline** to see the full workflow

```bash
python scripts/run_all.py
```

### Helpful Resources

- `docs/ml-core-foundation-framework.md` - Deep reference
- `docs/intern-free-tooling-guide.md` - Tooling guide
- `scripts/run_all.py` - Complete starter pipeline
- `work/notebooks/` - Your notebook skeletons

In [ ]:
print("\n? Ready to start your internship journey!")
print("Remember: Core idea first ? AI-assisted implementation ? Human judgment last")